# Session 20: Logistic Regression and How Models Learn

Data Science with Python, Level 1, Stage 3: Statistics and ML Fundamentals

## Learning Objectives

By the end of this session you will be able to:

1. Measure a model's error with a loss function.
2. Explain gradient descent and the learning rate.
3. Tell regression from classification.
4. Fit a logistic regression and read its probabilities.
5. Compare a model with a baseline.
6. Read a confusion matrix and choose a threshold.

## Where We Are

In Session 19 you called `fit` and scikit-learn found the best line. Today you look inside. A model measures how wrong it is with a **loss function**, then changes its numbers a little at a time in the direction that lowers the loss. This idea, **gradient descent**, is how almost every model learns, from a straight line to a neural network.

In the second half you predict a **category** instead of a number: will a customer leave or stay? Logistic regression turns the same weighted sum into a probability between 0 and 1. You will also learn why accuracy alone can fool you, and how to read a confusion matrix.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, SGDRegressor

sns.set_theme(style="whitegrid")
DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
exams = pd.read_csv(DATA_URL + "exam_scores.csv")
x = exams["study_hours_week"].to_numpy()
y = exams["math"].to_numpy(dtype=float)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
churn["left"] = (churn["churn"] == "Yes").astype(int)

## 1. Measuring Error with a Loss Function

A line predicts `y_hat = w * x + b`, where `w` is the slope (also called the **weight**) and `b` the intercept (the **bias**). To compare lines we need one number for the total error. Linear regression uses the **mean squared error** (MSE): the average of the squared residuals. Squaring makes every error positive and punishes big errors more.

In [ ]:
def mse(w, b):
    y_hat = w * x + b
    return np.mean((y - y_hat) ** 2)

for w, b in [(0, 57), (1, 50), (1.5, 46), (3, 30)]:
    print(f"w = {w}, b = {b}: loss = {mse(w, b):.1f}")

In [ ]:
ws = np.linspace(-1, 4, 100)
losses = [mse(w, 46.2) for w in ws]

plt.plot(ws, losses)
plt.title("Loss for different slopes (b fixed at 46.2)")
plt.xlabel("Slope w")
plt.ylabel("Mean squared error")
plt.show()

## 2. The Gradient

The **gradient** is the slope of the loss with respect to each parameter. If increasing `w` makes the loss go up, the gradient for `w` is positive, so we should **decrease** `w`. For MSE the gradients have simple formulas:

- for `w`: `-2 * mean(x * (y - y_hat))`
- for `b`: `-2 * mean(y - y_hat)`

You do not need to derive them. What matters is the rule: **step against the gradient**.

In [ ]:
def gradients(w, b):
    error = y - (w * x + b)
    dw = -2 * np.mean(x * error)
    db = -2 * np.mean(error)
    return dw, db

for w in [0.5, 3.0]:
    dw, db = gradients(w, 46.2)
    print(f"w = {w}: gradient for w {dw:.1f}, for b {db:.1f}")

## 3. Gradient Descent

In [ ]:
w, b = 0.0, 0.0
learning_rate = 0.002
history = []

for step in range(20_000):
    dw, db = gradients(w, b)
    w -= learning_rate * dw
    b -= learning_rate * db
    if step % 2000 == 0:
        history.append(mse(w, b))

print(f"w = {w:.3f}, b = {b:.2f}, loss = {mse(w, b):.1f}")

In [ ]:
sk = LinearRegression().fit(x.reshape(-1, 1), y)
print(f"scikit-learn: w = {sk.coef_[0]:.3f}, b = {sk.intercept_:.2f}")

## 4. The Learning Rate

In [ ]:
def train(lr, steps=300, w=0.0, b=46.2):
    losses = []
    for _ in range(steps):
        dw, db = gradients(w, b)
        w, b = w - lr * dw, b - lr * db
        losses.append(mse(w, b))
    return losses

for lr in [0.0005, 0.005, 0.017]:
    plt.plot(train(lr), label=f"learning rate {lr}")
plt.yscale("log")
plt.ylim(100, 5000)
plt.title("Loss during training")
plt.xlabel("Step")
plt.legend()
plt.show()

One more practical point before classification: when features have very different sizes, for example area in square metres next to the number of rooms, gradient descent is slow and zigzags. Scaling every feature to a similar size with `StandardScaler` fixes this, and you will see it in the churn model below.

## 5. From Numbers to Categories

In [ ]:
sns.stripplot(data=churn, x="tenure_months", y="churn",
              alpha=0.2, size=3)
plt.title("Leavers had shorter tenure")
plt.xlabel("Months as a customer")
plt.show()

## 6. The Sigmoid Function

Logistic regression first computes a weighted sum, exactly like linear regression: `z = w * x + b`. Then it passes `z` through the **sigmoid** function, `1 / (1 + e^(-z))`, which turns any number into a value between 0 and 1. Large positive `z` gives a probability close to 1; large negative `z` close to 0.

In [ ]:
z = np.linspace(-8, 8, 200)
sigmoid = 1 / (1 + np.exp(-z))

plt.plot(z, sigmoid)
plt.axhline(0.5, color="gray", linestyle="--")
plt.title("The sigmoid turns any number into a probability")
plt.xlabel("z = w * x + b")
plt.ylabel("Probability")
plt.show()

## 7. Fitting a Logistic Regression

In [ ]:
model = LogisticRegression()
model.fit(churn[["tenure_months"]], churn["left"])

new = pd.DataFrame({"tenure_months": [1, 12, 36, 60]})
new["P(leave)"] = model.predict_proba(new)[:, 1].round(3)
print(new)

## 8. Many Features and a Fair Test

In [ ]:
churn["monthly_contract"] = (churn["contract"] == "Month-to-month").astype(int)
churn["fiber"] = (churn["internet"] == "Fiber").astype(int)
churn["echeck"] = (churn["payment_method"] == "Electronic check").astype(int)

features = ["tenure_months", "monthly_charges", "support_calls", "senior",
            "monthly_contract", "fiber", "echeck"]
X = churn[features].astype(float)
y = churn["left"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

In [ ]:
scaler = StandardScaler().fit(X_train)
model = LogisticRegression()
model.fit(scaler.transform(X_train), y_train)

pred = model.predict(scaler.transform(X_test))
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("Baseline, always 'stays':", round(1 - y_test.mean(), 3))

## 9. Thresholds and the Confusion Matrix

In [ ]:
print(confusion_matrix(y_test, pred))

In [ ]:
proba = model.predict_proba(scaler.transform(X_test))[:, 1]

for t in [0.3, 0.5, 0.7]:
    tn, fp, fn, tp = confusion_matrix(y_test, (proba >= t).astype(int)).ravel()
    print(f"threshold {t}: caught {tp} of {tp + fn} leavers, {fp} false alarms")

## 10. Reading the Coefficients

In [ ]:
odds = pd.Series(np.exp(model.coef_[0]), index=features).sort_values()
print(odds.round(2))

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Loss explodes | A learning rate that is too large | Divide it by 10 and plot the loss again. |
| Fooled by accuracy | Reporting accuracy without a baseline | Compare with always predicting the most common class. |
| Leakage | Fitting the scaler on all the data before splitting | Fit the scaler on the training set only. |
| Fixed threshold | Always using 0.5 | Choose the threshold from the costs of each kind of error. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Plot the Descent

Run gradient descent on the standardised feature for 50 steps with learning rate 0.1, store the loss at every step, and plot it.

In [ ]:
# Write your code below

### In-Class Exercise 2: Find the Breaking Point

With the standardised feature, try learning rates 0.5, 0.9, 1.0 and 1.1 for 50 steps. Which ones converge and which explode?

In [ ]:
# Write your code below

### In-Class Exercise 3: Risky Customers

Using the model with all features from the lecture, list the ten test customers with the highest predicted risk, with their contract, tenure and whether they actually left.

In [ ]:
# Write your code below

### In-Class Exercise 4: Choose a Threshold

Suppose a missed leaver costs 300 and a wasted offer costs 40. For thresholds from 0.1 to 0.9 in steps of 0.1, compute the total cost on the test set. Which threshold is cheapest?

In [ ]:
# Write your code below

## Summary

- A loss function measures how wrong a model is; MSE averages squared errors.
- Gradient descent moves each parameter against its gradient, step by step.
- Too large a learning rate explodes; too small is very slow.
- Logistic regression passes a weighted sum through the sigmoid to get a probability.
- Always compare accuracy with a baseline that predicts the majority class.
- The confusion matrix and the threshold decide which mistakes you make.

## What's Next

Accuracy is only one way to score a model. Next session you will measure precision, recall, ROC curves and regression errors, and choose the metric that matches the real cost of each mistake.